In [ ]:
import scanpy as sc
import anndata as ad
import scvi
import sys
sys.path.append("/multiHIVE/src")
from multiHIVE import multiHIVE
import pandas as pd
import numpy as np

In [3]:
import torch, random
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed(0)

In [ ]:
adata = sc.read_h5ad("/Data/RNA_ADT/neurIPS/GSE194122_openproblems_neurips2021_cite_BMMC_processed.h5ad")

In [5]:
sc.pp.normalize_total(adata ,target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    flavor="seurat_v3",
    batch_key="batch",
    subset=True,
    layer="counts"
)
hvg = 2000

In [6]:
multiHIVE.setup_anndata(
    adata,
    layer="counts",
    batch_key="batch",
    protein_expression_obsm_key="protein_counts"
)

INFO     Using column names from columns of adata.obsm['protein_counts']                                           


/tmp/ipykernel_241925/1120891381.py:1: DeprecationWarning: multiHIVE is supposed to work with MuData. the use of anndata is deprecated and will be removed in scvi-tools 1.4. Please use setup_mudata
  multiHIVE.setup_anndata(


In [ ]:
vae = multiHIVE(adata, latent_distribution="normal", kl_dot_product=True, deep_network=False,
               n_genes=adata.shape[1],
    n_regions=0,
    n_proteins=134,)

In [ ]:
vae.train()


vae.get_latent_representation()

generated_data = vae.posterior_predictive_sample(adata, swap_latent=False)
hvg = 2000
rna_sample = generated_data[:,:hvg].copy()
proteins_sample = pd.DataFrame(generated_data[:,hvg:],  index= adata.obs_names, columns = adata.obsm['protein_counts'].columns)
adata.obsm['RNA_Z1_denoised'] = rna_sample
adata.obsm['protein_Z1_denoised'] = proteins_sample

generated_data = vae.posterior_predictive_sample(adata, swap_latent=True)
rna_sample = generated_data[:,:hvg].copy()
proteins_sample = pd.DataFrame(generated_data[:,hvg:],  index= adata.obs_names, columns = adata.obsm['protein_counts'].columns)
adata.obsm['RNA_Z2_denoised'] = rna_sample
adata.obsm['protein_Z2_denoised'] = proteins_sample

In [10]:
vae.save("./outputs/neurips/saved_model/", overwrite=True)

In [14]:
adata.write("./outputs/neurips/multiHIVE.h5ad")